# BÀI TẬP LỚN NHẬP MÔN HỌC MÁY (2026–2027)
## ĐỀ TÀI 14: DỰ ĐOÁN KHÁCH HÀNG HỦY ĐẶT PHÒNG KHÁCH SẠN
### NOTEBOOK 02: TIỀN XỬ LÝ DỮ LIỆU & PHÒNG CHỐNG RÒ RỈ (PREPROCESSING & ANTI-LEAKAGE)

---
### Nội dung chính:
1. Phân chia tập dữ liệu Train/Test (80% / 20%, Stratified Split, Seed = 42).
2. Xử lý triệt để rò rỉ dữ liệu (Loại bỏ `reservation_status` và `reservation_status_date`).
3. Xây dựng **Pipeline A (Cơ bản)**: Imputation + OneHotEncoder + StandardScaler.
4. Xây dựng **Pipeline B (Cải tiến)**: Kết hợp Feature Engineering + Gom nhóm danh mục hiếm + RobustScaler.
5. Kiểm chứng tính độc lập: Các tham số thống kê chỉ được học trên tập Train.


In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from src.config import RAW_DATA_PATH, LEAKAGE_COLUMNS, RANDOM_STATE, TEST_SIZE
from src.data_loader import load_features_and_target, get_train_test_split
from src.preprocessing import get_pipeline_a_preprocessor, get_pipeline_b_preprocessor

print("Đã nạp thành công các module tiền xử lý!")


Đã nạp thành công các module tiền xử lý!


## 1. Nạp dữ liệu và Phân tách Chống Data Leakage

In [2]:
X, y = load_features_and_target(drop_leakage=True)
print("Kích thước ma trận đặc trưng X:", X.shape)
print("Phân phối biến mục tiêu y:")
print(y.value_counts(normalize=True))

X_train, X_test, y_train, y_test = get_train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
print(f"Số mẫu Train: {len(X_train):,} | Số mẫu Test: {len(X_test):,}")


2026-11-17 02:15:24,376 - INFO - Đang đọc dữ liệu từ: D:\NH-M-8-Machine-Learning\data\raw\hotel_bookings.csv


2026-11-17 02:15:24,564 - INFO - Đọc dữ liệu thành công! Kích thước: 119,390 dòng x 32 cột. Bộ nhớ ước tính: 29.15 MB


2026-11-17 02:15:24,584 - INFO - [CHỐNG DATA LEAKAGE] Loại bỏ các cột gây rò rỉ thông tin tương lai: ['reservation_status', 'reservation_status_date']


2026-11-17 02:15:24,603 - INFO - Phân chia Train/Test: 80% Train, 20% Test (stratify=y, seed=42)


2026-11-17 02:15:24,677 - INFO - Kích thước tập huấn luyện (Train): 95,512 mẫu | Tập kiểm thử (Test): 23,878 mẫu


Kích thước ma trận đặc trưng X: (119390, 29)
Phân phối biến mục tiêu y:
is_canceled
0    0.629584
1    0.370416
Name: proportion, dtype: float64
Số mẫu Train: 95,512 | Số mẫu Test: 23,878


## 2. Thử nghiệm Pipeline A (Baseline Preprocessing)

In [3]:
preprocessor_a = get_pipeline_a_preprocessor(scale_numeric=True)
X_train_proc_a = preprocessor_a.fit_transform(X_train)
X_test_proc_a = preprocessor_a.transform(X_test)

print("Pipeline A hoàn tất biến đổi!")
print(f"Kích thước X_train sau Pipeline A: {X_train_proc_a.shape}")
print(f"Kích thước X_test sau Pipeline A: {X_test_proc_a.shape}")


Pipeline A hoàn tất biến đổi!
Kích thước X_train sau Pipeline A: (95512, 252)
Kích thước X_test sau Pipeline A: (23878, 252)


## 3. Thử nghiệm Pipeline B (Improved Preprocessing)

In [4]:
preprocessor_b = get_pipeline_b_preprocessor(scale_numeric=True)
X_train_proc_b = preprocessor_b.fit_transform(X_train)
X_test_proc_b = preprocessor_b.transform(X_test)

print("Pipeline B hoàn tất biến đổi!")
print(f"Kích thước X_train sau Pipeline B: {X_train_proc_b.shape}")
print(f"Kích thước X_test sau Pipeline B: {X_test_proc_b.shape}")


Pipeline B hoàn tất biến đổi!
Kích thước X_train sau Pipeline B: (95512, 88)
Kích thước X_test sau Pipeline B: (23878, 88)


## 4. Đánh giá sự khác biệt giữa hai Pipeline
- **Pipeline A:** Giữ nguyên các đặc trưng ban đầu, One-Hot toàn bộ danh mục, dẫn đến số chiều tăng mạnh (đặc biệt do cột `country`).
- **Pipeline B:** Bổ sung các đặc trưng phái sinh nghiệp vụ (`total_nights`, `total_guests`, `room_type_changed`, v.v.), đồng thời gom nhóm danh mục hiếm (min_frequency = 0.01) giúp kiểm soát bùng nổ số chiều và tăng cường khả năng tổng quát hóa của mô hình.
